# Chapter 8 — What would we have missed? Injection and recovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/halo-substructure-tutorial/blob/main/notebooks/08_injection_recovery.ipynb)

**Goal.** A list of detections means little without knowing which satellites the search *could* have found. Here we
create synthetic satellites of given luminosity and distance, inject their stars into a real SDSS patch, run the
unchanged search of Chapter 5, and measure the fraction recovered. This is the recovery efficiency — the basis of
every estimate of how many satellites the Milky Way has (Koposov et al. 2008, https://arxiv.org/abs/0706.2687;
Drlica-Wagner et al. 2020, https://doi.org/10.3847/1538-4357/ab7eb9).

**You will learn**
1. to simulate the resolved stars of a satellite from an isochrone and an initial mass function, normalised to a total
   absolute magnitude $M_V$;
2. to give them realistic photometric errors and the catalogue's magnitude limit;
3. to measure the recovery efficiency on a grid of $(M_V, D)$, with binomial uncertainties.

**What this does not include.** We add synthetic stars to a *catalogue*. Real satellite stars must also be detected
in the images, deblended and classified as point sources; near the faint limit some would be lost (Chapter 1). The
efficiency measured here is conditional on the catalogue selection and is an upper limit to the true survey efficiency.

In [ ]:
import hashlib, json, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy import special, stats

DATA_URL = 'https://github.com/vasilybelokurov/halo-substructure-tutorial/releases/download/data-v0.1/'
_local = Path('../data/MANIFEST.json')            # present when you run inside a clone of the repository
MANIFEST = json.loads(_local.read_text() if _local.exists() else urllib.request.urlopen(DATA_URL + 'MANIFEST.json').read())


def load(name, cache=Path('data')):
    """Download <name>.parquet from the tutorial's data release once, check its SHA-256, return a DataFrame."""
    path = cache / f'{name}.parquet'
    if not path.exists():
        cache.mkdir(exist_ok=True)
        print(f'downloading {path.name} ...')
        urllib.request.urlretrieve(DATA_URL + path.name, path)
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    assert sha == MANIFEST[name]['sha256'], f'{path} does not match the manifest: delete it and run again'
    return pd.read_parquet(path)


def tangent(ra, dec, ra0, dec0):
    """Gnomonic (tangent-plane) projection about (ra0, dec0). Returns (xi, eta) in degrees: xi towards east, eta north."""
    ra, dec, ra0, dec0 = map(np.radians, (ra, dec, ra0, dec0))
    cosc = np.sin(dec0) * np.sin(dec) + np.cos(dec0) * np.cos(dec) * np.cos(ra - ra0)
    xi = np.cos(dec) * np.sin(ra - ra0) / cosc
    eta = (np.cos(dec0) * np.sin(dec) - np.sin(dec0) * np.cos(dec) * np.cos(ra - ra0)) / cosc
    return np.degrees(xi), np.degrees(eta)


lvdb = load('lvdb').set_index('key')
iso = load('isochrones_sdss')

## 8.1 The search of Chapter 5, unchanged

The functions below are copied from Chapter 5: CMD masks for a grid of distance moduli, and Poisson significance
maps for four kernel sizes. Nothing is tuned here.

In [ ]:
cb, mb = np.arange(-0.4, 1.21, 0.04), np.arange(16, 22.51, 0.1)
cc, mm = 0.5 * (cb[1:] + cb[:-1]), 0.5 * (mb[1:] + mb[:-1])
CC, MM = np.meshgrid(cc, mm, indexing='ij')
I = iso[np.isclose(iso.logage, 10.1) & np.isclose(iso.feh, -2.1) & (iso.stage <= 7)].sort_values('m_ini')
_c, _g = (I.g - I.r).to_numpy(), I.g.to_numpy()
_arc = np.concatenate([[0], np.cumsum(np.hypot(np.diff(_c), np.diff(_g)))])
_s = np.linspace(0, _arc[-1], 3000)
ISO_C, ISO_G, ISO_W = np.interp(_s, _arc, _c), np.interp(_s, _arc, _g), np.gradient(np.interp(_s, _arc, I.int_imf))
MUS = np.arange(16.5, 21.51, 0.5)
SIGMAS = np.array([1, 2, 4, 8])


def satellite_cmd(mu, err=lambda m: 0.02 + 0.08 * np.exp((m - 22) / 1.0)):
    pg = ISO_G + mu
    sc, sm = np.hypot(np.sqrt(2) * err(pg), 0.03), np.hypot(err(pg), 0.1)
    S = np.einsum('k,kij->ij', ISO_W, np.exp(-0.5 * ((CC - ISO_C[:, None, None]) / sc[:, None, None])**2
                                             - 0.5 * ((MM - pg[:, None, None]) / sm[:, None, None])**2) / sc[:, None, None] / sm[:, None, None])
    return S / S.sum()


def cmd_index(c, m):
    i, j = np.digitize(c, cb) - 1, np.digitize(m, mb) - 1
    ok = (i >= 0) & (i < len(cc)) & (j >= 0) & (j < len(mm))
    return np.where(ok, i * len(mm) + j, -1)


def cmd_mask(S, B, keep=0.5):
    order = np.argsort((S / B).ravel())[::-1]
    cs, cbk = np.cumsum(S.ravel()[order]), np.cumsum(B.ravel()[order])
    M = np.zeros(S.size, bool)
    M[order[:np.argmax(np.where(cs >= keep, cs / np.sqrt(cbk), 0)) + 1]] = True
    return M.reshape(S.shape)


def make_masks(c, m):
    """CMD masks (one per trial distance modulus), built from the field CMD of the patch."""
    H = gaussian_filter(np.histogram2d(c, m, [cb, mb])[0], 1.0)
    B = np.maximum(H, 1.0) / np.maximum(H, 1.0).sum()
    return np.array([cmd_mask(satellite_cmd(mu), B).ravel() for mu in MUS])


def search(x, y, w, mask, extent, pix=1 / 60, sig_bg=30):
    xe, ye = np.arange(extent[0], extent[1] + pix / 2, pix), np.arange(extent[2], extent[3] + pix / 2, pix)
    M = mask.astype(float)
    norm_bg = gaussian_filter(M, sig_bg)
    out = np.full((len(w), len(SIGMAS)) + mask.shape, np.nan)
    for a, wa in enumerate(w):
        W = np.histogram2d(x, y, [xe, ye], weights=wa)[0]
        Wb = gaussian_filter(W, sig_bg) / np.maximum(norm_bg, 1e-9)
        for b, s in enumerate(SIGMAS):
            norm = gaussian_filter(M, s)
            area = 4 * np.pi * s**2 * norm
            n = gaussian_filter(W, s) / np.maximum(norm, 1e-9) * area
            lam = np.maximum(Wb * area, 1e-9)
            S = stats.norm.isf(np.clip(special.gammainc(np.maximum(n, 1e-9), lam), 1e-300, 1))
            out[a, b] = np.where(mask & (norm > 0.5) & (n > lam), S, np.where(mask & (norm > 0.5), 0.0, np.nan))
    return out

## 8.2 The field: the Segue 1 patch without Segue 1

We use the 1.5° SDSS patch around Segue 1, search only its inner 1° (as in Chapter 5), and exclude the disc within
0.25° of Segue 1 itself, so that the real satellite is not counted as a recovery. We first calibrate the detection
threshold: the 99th percentile of the maximum S in null patches (positions shuffled, Chapter 5).

In [ ]:
ref = lvdb.loc['segue_1']
cat = load('segue1_sdss')
x0, y0 = tangent(cat.ra.to_numpy(), cat.dec.to_numpy(), ref.ra, ref.dec)
g0 = (cat.psfmag_g - cat.extinction_g).to_numpy()
r0 = (cat.psfmag_r - cat.extinction_r).to_numpy()
c0 = g0 - r0
masks = make_masks(c0, g0)
rmax = 1.48
extent = (-rmax, rmax, -rmax, rmax)
pc = np.arange(-rmax, rmax, 1 / 60) + 1 / 120
PX, PY = np.meshgrid(pc, pc, indexing='ij')
footprint = np.hypot(PX, PY) < rmax
searched = (np.hypot(PX, PY) < rmax - 0.5) & (np.hypot(PX, PY) > 0.25)


def run(x, y, c, m):
    """Search a star list; return (max S, x, y) over the searched region."""
    k = cmd_index(c, m)
    w = np.where(k >= 0, masks[:, np.maximum(k, 0)], False).astype(float)
    S = np.where(searched[None, None], search(x, y, w, footprint, extent), np.nan)
    a, b, i, j = np.unravel_index(np.nanargmax(S), S.shape)
    return S[a, b, i, j], pc[i], pc[j]


rng = np.random.default_rng(8)
null = []
for t in range(100):
    rr, th = rmax * np.sqrt(rng.uniform(0, 1, len(x0))), rng.uniform(0, 2 * np.pi, len(x0))
    null.append(run(rr * np.cos(th), rr * np.sin(th), c0, g0)[0])
threshold = np.percentile(null, 99)
real = run(x0, y0, c0, g0)
print(f'null maxima: median {np.median(null):.2f}, 99th percentile (threshold) {threshold:.2f}; '
      f'real patch outside Segue 1: max S = {real[0]:.2f}')

## 8.3 Synthetic satellites

A satellite of absolute magnitude $M_V$ at distance $D$:

1. **Stars.** Draw stars from the isochrone (12.6 Gyr, [Fe/H] = −2.1) with probabilities proportional to the IMF
   (sampling uniformly in the cumulative IMF, `int_imf`), and keep drawing until their total V-band luminosity reaches
   $M_V$. V is computed from g and r with the stellar transformation $V = g - 0.59(g-r) - 0.01$ (Jester et al. 2005,
   https://ui.adsabs.harvard.edu/abs/2005AJ....130..873J). Most of the light comes from a few giants, so two satellites with
   the same $M_V$ can have quite different numbers of detectable stars.
2. **Distance and errors.** Add $\mu = 5\log_{10}(D/10\,{\rm pc})$; add Gaussian errors with the median SDSS error at
   each magnitude (measured from the patch); keep stars with $g < 23$, the catalogue limit.
3. **Positions.** An exponential profile with half-light radius $r_h = 50$ pc (so $\theta_h = r_h/D$), round, at a random
   position in the searched region.

In [ ]:
import warnings
warnings.filterwarnings('ignore', 'Mean of empty slice')      # magnitude bins without stars give NaN, filled below
eb = np.arange(14, 23.51, 0.25)
em = 0.5 * (eb[1:] + eb[:-1])
med_err = {b: np.array([np.nanmedian(cat[f'psfmagerr_{b}'][(cat[f'psfmag_{b}'] >= lo) & (cat[f'psfmag_{b}'] < hi)])
                        for lo, hi in zip(eb[:-1], eb[1:])]) for b in 'gr'}
sigma = lambda b, m: np.interp(m, em, np.nan_to_num(med_err[b], nan=0.25))
cum = I.int_imf.to_numpy()
V_iso = I.g.to_numpy() - 0.59 * (I.g - I.r).to_numpy() - 0.01


def draw_satellite(MV, D_kpc, rh_pc=50):
    """Observed (x, y [deg], g, r) of the detectable stars of a synthetic satellite, centred at (0, 0)."""
    L_target, L, gs, rs = 10**(-0.4 * MV), 0.0, [], []
    while L < L_target:
        u = rng.uniform(cum[0], cum[-1], 2000)
        g, r, V = np.interp(u, cum, I.g), np.interp(u, cum, I.r), np.interp(u, cum, V_iso)
        Lc = np.cumsum(10**(-0.4 * V))
        n = np.searchsorted(Lc, L_target - L) + 1                     # stop when the target luminosity is reached
        gs.append(g[:n]); rs.append(r[:n]); L += Lc[min(n, len(Lc)) - 1]
    g, r = np.concatenate(gs), np.concatenate(rs)
    mu = 5 * np.log10(D_kpc * 100)
    g, r = g + mu, r + mu
    g, r = g + rng.normal(size=g.size) * sigma('g', g), r + rng.normal(size=r.size) * sigma('r', r)
    keep = g < 23
    a = np.degrees(rh_pc / (D_kpc * 1e3)) / 1.678
    rad, phi = rng.gamma(2, a, keep.sum()), rng.uniform(0, 2 * np.pi, keep.sum())
    return rad * np.cos(phi), rad * np.sin(phi), g[keep], r[keep]


xs, ys, gs, rs = draw_satellite(-4, 50)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.3))
ax[0].scatter(x0 * 60, y0 * 60, s=1, c='0.7')
ax[0].scatter(xs * 60 + 30, ys * 60 + 20, s=4, c='C3')
ax[0].set(xlim=(60, 0), ylim=(-10, 50), aspect='equal', xlabel="ξ [arcmin]", ylabel="η [arcmin]",
          title=f'M_V = −4 at 50 kpc: {len(gs)} stars with g < 23')
ax[1].scatter(g0 - r0, g0, s=1, c='0.7')
ax[1].scatter(gs - rs, gs, s=6, c='C3')
ax[1].set(xlim=(-0.5, 1.3), ylim=(23, 15), xlabel='$(g-r)_0$', ylabel='$g_0$')
plt.tight_layout()

**Figure 8.1.** One synthetic satellite ($M_V = -4$, 50 kpc), injected into the field: on the sky (left) and in the
CMD (right). Only its giants, horizontal-branch stars and brightest turn-off stars are above the SDSS limit.

## 8.4 Recovery efficiency

For each cell of a grid in $M_V$ and $D$ we inject 25 satellites, one at a time, at random positions 0.35°–0.9° from the
patch centre, and search. A satellite is **recovered** if the maximum of the search lies within max(2′, $\theta_h$) of the
injected centre and exceeds the threshold. With 25 injections the binomial uncertainty is ±0.1 near 50% efficiency.

In [ ]:
MVs = np.array([-1, -2, -3, -4, -5, -6])
Ds = np.array([25, 50, 100, 200])
n_inj = 25
eff = np.zeros((len(MVs), len(Ds)))
for a, MV in enumerate(MVs):
    for b, D in enumerate(Ds):
        hit = 0
        for t in range(n_inj):
            xs, ys, gs, rs = draw_satellite(MV, D)
            rad, th = rng.uniform(0.35, 0.9), rng.uniform(0, 2 * np.pi)
            cx, cy = rad * np.cos(th), rad * np.sin(th)
            Smax, px, py = run(np.concatenate([x0, xs + cx]), np.concatenate([y0, ys + cy]),
                               np.concatenate([c0, gs - rs]), np.concatenate([g0, gs]))
            tol = max(2 / 60, np.degrees(50 / (D * 1e3)))
            hit += (Smax > threshold) and (np.hypot(px - cx, py - cy) < tol)
        eff[a, b] = hit / n_inj
print(pd.DataFrame(eff, index=[f'M_V={m}' for m in MVs], columns=[f'{d} kpc' for d in Ds]).to_string())

fig, ax = plt.subplots(figsize=(6.5, 4.5))
im = ax.imshow(eff, origin='lower', cmap='viridis', vmin=0, vmax=1, aspect='auto',
               extent=[-0.5, len(Ds) - 0.5, -0.5, len(MVs) - 0.5])
for a in range(len(MVs)):
    for b in range(len(Ds)):
        ax.text(b, a, f'{eff[a, b]:.2f}', ha='center', va='center', color='w' if eff[a, b] < 0.6 else 'k')
ax.set(xticks=range(len(Ds)), xticklabels=Ds, yticks=range(len(MVs)), yticklabels=MVs,
       xlabel='distance [kpc]', ylabel='$M_V$', title='recovery efficiency, SDSS, $r_h$ = 50 pc')
plt.colorbar(im, label='fraction recovered')
plt.tight_layout()
assert eff[list(MVs).index(-6), list(Ds).index(50)] > 0.9
assert eff[list(MVs).index(-2), list(Ds).index(100)] < 0.2

**Figure 8.2.** Recovery efficiency of the Chapter 5 search in SDSS for round satellites with $r_h = 50$ pc. Bright
nearby satellites are always found; satellites fainter than $M_V \approx -3$ are found only within a few tens of kpc.
The transition is gradual because of the stochastic number of giants. Compare with the real satellites: Segue 1
($M_V = -1.3$, 23 kpc, $r_h \approx 24$ pc) lies near the edge of SDSS's reach — and it was found.

The efficiency also depends on size (a larger satellite at the same $M_V$ has lower surface brightness), on the
stellar population, on the local star density and on the depth. A full selection function maps all of these
(Drlica-Wagner et al. 2020); and, as noted above, catalogue-level injection omits losses in detection and
star–galaxy classification.

## Summary

- Synthetic satellites drawn from an isochrone and an IMF, with the survey's errors and limit, measure what a search can find.
- Recovery requires both a calibrated threshold (from nulls) and a positional match.
- The efficiency is a strong function of $M_V$ and distance; the faintest known satellites are found only nearby.